**Eye-Tracking & Neurocognitive Intelligence | Eye-Tracking Foundations & Signal Analysis | Presented by: Reza Saadatyar (2026)**

**07 | Integrated Evidence and Reproducibility Review**

![Conceptual workflow for stage 07](../../images/07_integrated_evidence_and_reproducibility.svg)

**Objective:** Integrate existing stage-owned quality, event, feature, statistical and sensitivity summaries without reprocessing data.

**Dataset:** Current numerical tables generated by Notebooks 01, 03, 04, 05 and 06.

**Notebook scope:** Cheap read-only synthesis of saved evidence and explicit limitations; no model training.

**Execution status:** Source code prepared. No original-data results are claimed until this notebook is executed with the specified release.


**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_tracking_foundations")

# ---------------------------- 2. Standard Library ----------------------------
from pathlib import Path

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

# ------------------------ 5. Study Research Utilities ------------------------
from core.stage_outputs import ensure_stage_inputs
from gaze_source import read_stage

# --------------------- 6. Output and Presentation Setup ----------------------
output_dir = root / "eye_tracking_foundations" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(
    font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5,
    label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(10, 4),
)
print("Current outputs:", output_dir)


**02. Recover Current Stage-Owned Evidence**

For every required stage, use the exact named outputs or its explicitly
named ZIP from the preceding runtime. No raw source, detector or feature
extractor is invoked in this notebook.


In [ ]:
# ================= 02. Recover Current Stage-Owned Evidence ==================

REQUIRED={
    1:("01_quality_by_record.csv",),
    3:("03_vendor_agreement.csv",),
    4:("04_features_by_record.csv",),
    5:("05_descriptive_statistics.csv",),
    6:("06_sensitivity_summary.csv",),
}
for stage, names in REQUIRED.items():
    ensure_stage_inputs(output_dir,stage,names)
quality=read_stage(output_dir/"01_quality_by_record.csv",("record_id","valid_fraction"))
vendor=read_stage(output_dir/"03_vendor_agreement.csv",("record_id","sample_agreement"))
features=read_stage(output_dir/"04_features_by_record.csv",("record_id","task","participant_id"))
stats=read_stage(output_dir/"05_descriptive_statistics.csv",("task","feature"))
sensitivity=read_stage(output_dir/"06_sensitivity_summary.csv",("threshold_dva_s",))


**03. Check Record-Level Evidence Correspondence**

A shared record set is essential when synthesising quality, event and
feature outputs. If a producer was rerun on another subset, stop rather
than silently combine incompatible reported summaries.


In [ ]:
# ============== 03. Check Record-Level Evidence Correspondence ===============

sets={
    "quality":set(quality.record_id),
    "vendor":set(vendor.record_id),
    "features":set(features.record_id),
}
if len({frozenset(s) for s in sets.values()})!=1:
    raise ValueError("Stage evidence refers to incompatible record sets; regenerate affected downstream outputs")
if not set(features.participant_id.astype(str).str.zfill(3)).issubset(
        set(quality.participant_id.astype(str).str.zfill(3))):
    raise ValueError("Participant identities do not align across stage outputs")
print("Aligned records:",len(sets["quality"]))


**04. Prepare a Bounded Evidence Overview**

This summary distinguishes observed data coverage, vendor-comparison
availability, event-derived features and sensitivity. It does not claim
independent reproduction or performance outside the selected source subset.


In [ ]:
# ================== 04. Prepare a Bounded Evidence Overview ==================

summary=pd.DataFrame([{
    "dataset":"GazeBase", "selected_records":len(quality),
    "independent_participants":features.participant_id.nunique(),
    "mean_record_valid_fraction":quality.valid_fraction.mean(),
    "records_with_vendor_comparison":int(vendor.sample_agreement.notna().sum()),
    "n_record_feature_rows":len(features),
    "n_fixed_thresholds_evaluated":len(sensitivity),
    "input_provenance":"operator_supplied_GazeBase_CSV_not_independently_verified",
    "scope":"bounded_descriptive_subset_not_generalisation",
}])
summary.to_csv(output_dir/"07_evidence_summary.csv",index=False)
display(summary)


**05. Render Summary from Saved Numerical Tables**

Only already computed output tables feed this plot. The synthesis does not
re-run any scientific stage. The plotted values describe the selected subset.


In [ ]:
# ============== 05. Render Summary from Saved Numerical Tables ===============

fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].hist(quality.valid_fraction.dropna(),bins=min(10,max(1,len(quality))))
axes[0].set(xlabel="Record valid fraction",ylabel="Records",title="Input quality")
axes[1].plot(sensitivity.threshold_dva_s,
             sensitivity.mean_predicted_fixation_fraction,marker="o")
axes[1].set(xlabel="I-VT threshold (dva/s)",ylabel="Mean fixation fraction",
            title="Method sensitivity",ylim=(0,1))
fig.tight_layout()
save_figure(fig,output_dir/"07_study_overview.png")
plt.show()


**06. Interpretation, Scope and Export**

Report these as descriptive outputs from an explicitly selected set of
source recordings. No event detector has been independently validated,
and no cognitive, clinical or participant-generalisation result follows.
Save a final per-stage ZIP for manual review and later documentation updates.


In [ ]:
# =================== 06. Interpretation, Scope and Export ====================

print("Synthesis depends only on saved stage files; no acquisition or fitting was rerun.")
download_outputs_from_colab("eye_tracking_foundations/output","07")
